# Clase 13 — Consumir APIs con `requests`

## Objetivos de la clase

- Entender el ciclo **request / response** de HTTP: qué mandás, qué te contestan.
- Hacer pedidos GET y POST con la librería `requests` desde Python.
- Leer el `status_code`, los `headers` y el cuerpo de una respuesta.
- Convertir una respuesta JSON en un DataFrame de pandas, incluyendo el caso de JSON **anidado** con `pd.json_normalize()`.
- Consumir una API pública real (sin autenticación) durante la clase.

> Nota técnica: este notebook hace pedidos HTTP reales a internet. En Google Colab `requests` ya viene instalado. Si lo corrés en Jupyter local y no lo tenés, descomentá `# !pip install requests` en la primera celda de código.

## ¿Por qué esto importa?

Hasta ahora trabajaste con datos que ya tenías guardados: un CSV, un diccionario escrito a mano, un archivo que te pasaron. Pero la mayoría de los datos interesantes del mundo real **no viven en tu disco** — viven en el servidor de otra empresa y cambian todo el tiempo (el clima, una cotización, un resultado deportivo). Para conseguirlos, tenés que pedirlos en el momento.

Ahí entra una **API** (Application Programming Interface): la puerta que un servicio deja abierta para que tu programa le pida datos sin tener que entender cómo funciona por dentro. Vos no sabés cómo la NBA guarda sus estadísticas ni cómo un banco calcula el tipo de cambio — solo necesitás saber **qué pedir** y **qué formato de respuesta esperar**.

Dato curioso: ya usaste una API sin saberlo. Pandas es una biblioteca que expone una interfaz (`.head()`, `.mean()`, `pd.DataFrame(...)`) sin que necesites saber cómo está resuelta por dentro. Consumir una API en internet es exactamente la misma idea, un escalón más allá.

## Cómo funciona HTTP: pedido y respuesta

Pensalo como pedir comida en un restaurante: vos (el **cliente**) le decís al mozo qué querés en un formato que ambos entienden (el **request**), y el mozo te trae un plato o te avisa que no hay (la **response**). Eso es HTTP.

Toda esta comunicación viaja sobre una **URL**, que se descompone en tres partes:

| Parte | Qué es | Ejemplo |
|---|---|---|
| Esquema | El protocolo | `https://` |
| Dirección base | Dónde vive el servidor | `official-joke-api.appspot.com` |
| Ruta | Qué recurso pedís puntualmente | `/jokes/ten` |

Cuando esa ruta corresponde a una acción del servidor (no un archivo estático) se la llama **endpoint**.

El request lleva un **método HTTP** que le dice al servidor qué querés hacer:

- **GET**: pedís datos (es lo que hace tu navegador todo el tiempo).
- **POST**: enviás datos para que el servidor los procese o guarde.

La respuesta trae un **código de estado**:

| Rango | Significado | Ejemplo |
|---|---|---|
| 2xx | Éxito | `200 OK` |
| 3xx | Redirección | El recurso se movió |
| 4xx | Error tuyo | `404 Not Found` |
| 5xx | Error del servidor | Algo falló procesando tu pedido |

Y ambos lados llevan **headers**: metadata sobre el intercambio (tipo de contenido, fecha, codificación...), no el contenido en sí.

In [ ]:
# En Google Colab requests ya viene instalado.
# Si estás en Jupyter local puro y no lo tenés, descomentá la siguiente línea:
# !pip install requests

import requests
import pandas as pd

## GET con `requests`: el primer pedido real

Vamos a pedirle datos a una API pública que no necesita clave de autenticación: la **Official Joke API**, que devuelve chistes en inglés en formato JSON.

`requests.get(url)` hace el pedido y te devuelve un objeto respuesta (acá lo llamamos `r`) con todo el intercambio: código de estado, headers, cuerpo.

Como en clase dependemos de la red, envolvemos el pedido en un `try/except`: si algo falla (sin wifi, la API está caída), no queremos que el notebook explote — seguimos con un aviso.

In [ ]:
url_chistes = "https://official-joke-api.appspot.com/jokes/ten"

try:
    r = requests.get(url_chistes, timeout=10)
    print("status_code:", r.status_code)
except requests.exceptions.RequestException as e:
    r = None
    print("No se pudo conectar a la API:", e)
    print("Seguimos con datos de ejemplo más abajo.")

## Headers: request y response

`r.request` guarda el pedido que vos mandaste (con sus propios headers y, en un GET, sin body). `r.headers` guarda lo que contestó el servidor — se comporta como un diccionario de Python.

In [ ]:
if r is not None:
    print("Headers que MANDASTE vos (request):")
    print(r.request.headers)
    print()
    print("Body del request (en un GET normal, no hay body):", r.request.body)
    print()
    print("Headers que MANDÓ el servidor (response):")
    print(r.headers)
    print()
    print("Content-Type de la respuesta:", r.headers.get("Content-Type"))
else:
    print("Sin conexión: no hay objeto de respuesta para inspeccionar en esta corrida.")

## De JSON a DataFrame: `.json()` + `pd.DataFrame()`

El método `.json()` del objeto respuesta convierte el cuerpo (texto JSON) directamente en listas/diccionarios de Python — es equivalente a hacer `json.loads(r.text)`, pero en un solo paso.

Como buena práctica, **siempre revisá el `status_code` antes de confiar en los datos**: si el pedido falló, `.json()` puede no tener lo que esperás (o directamente tirar error). Por eso, si `r` es `None` o el `status_code` no es 200, usamos un JSON de respaldo (`fallback`) hardcodeado — así el notebook funciona igual si no hay internet en el momento de correrlo.

In [ ]:
# JSON de respaldo por si falla la conexión durante la demo
chistes_fallback = [
    {"type": "general", "setup": "¿Por qué las focas del circo miran siempre hacia arriba?",
     "punchline": "Porque ahí están los entrenadores.", "id": 1},
    {"type": "general", "setup": "¿Qué le dice un semáforo a otro?",
     "punchline": "No me mires que me estoy cambiando.", "id": 2},
    {"type": "programming", "setup": "¿Cuántos programadores hacen falta para cambiar una lamparita?",
     "punchline": "Ninguno, es un problema de hardware.", "id": 3},
]

if r is not None and r.status_code == 200:
    chistes = r.json()
    print("Datos obtenidos en vivo de la API ✅")
else:
    chistes = chistes_fallback
    print("Usando datos de ejemplo (fallback) ⚠️")

chistes[:2]

In [ ]:
df_chistes = pd.DataFrame(chistes)
df_chistes.head()

## JSON anidado: `pd.json_normalize()`

El caso anterior era un JSON **plano**: cada chiste es un diccionario simple, así que `pd.DataFrame()` lo resuelve directo. Pero muchas APIs devuelven JSON **anidado** — un diccionario adentro de otro. Ahí `pd.DataFrame()` no alcanza: la columna anidada queda como una columna de diccionarios, inútil para filtrar.

Ejemplo real: la API de **Fruityvice** devuelve, para cada fruta, un diccionario `nutritions` con sus propias claves (`calories`, `fat`, `sugar`...).

In [ ]:
# JSON de respaldo (estructura idéntica a la real, anidada) por si falla la conexión
frutas_fallback = [
    {"name": "Cherry", "id": 1, "family": "Rosaceae", "genus": "Prunus", "order": "Rosales",
     "nutritions": {"calories": 50, "fat": 1.0, "sugar": 8.0, "carbohydrates": 12.0, "protein": 1.0}},
    {"name": "Banana", "id": 2, "family": "Musaceae", "genus": "Musa", "order": "Zingiberales",
     "nutritions": {"calories": 96, "fat": 0.2, "sugar": 17.2, "carbohydrates": 22.0, "protein": 1.0}},
]

try:
    r_frutas = requests.get("https://fruityvice.com/api/fruit/all", timeout=10)
    if r_frutas.status_code == 200:
        frutas = r_frutas.json()
        print("Datos en vivo ✅")
    else:
        frutas = frutas_fallback
        print(f"La API contestó {r_frutas.status_code}, usando fallback ⚠️")
except requests.exceptions.RequestException as e:
    frutas = frutas_fallback
    print("No se pudo conectar, usando fallback ⚠️:", e)

In [ ]:
# Si lo pasamos directo a DataFrame, 'nutritions' queda como columna de diccionarios
df_frutas_plano = pd.DataFrame(frutas)
df_frutas_plano.head()

In [ ]:
# pd.json_normalize() aplana los diccionarios anidados en columnas propias:
# 'nutritions' -> 'nutritions.calories', 'nutritions.fat', etc.
df_frutas = pd.json_normalize(frutas)
df_frutas.head()

In [ ]:
# Con el JSON ya aplanado, filtrar es tan directo como con cualquier DataFrame
cereza = df_frutas.loc[df_frutas["name"] == "Cherry"]
cereza.iloc[0]["family"], cereza.iloc[0]["nutritions.calories"]

**Regla práctica:** si el JSON es plano, `pd.DataFrame()` alcanza. Si tiene diccionarios anidados dentro de las claves, usá `pd.json_normalize()` desde el principio — te ahorra desarmar la estructura vos mismo con bucles.

## GET con parámetros y POST: mandar información

Un GET puede llevar parámetros en la URL (la *query string*: todo después del `?`). En Python no armás esa cadena a mano: le pasás un diccionario al argumento `params` de `requests.get()`.

El método **POST**, en cambio, manda los datos en el **cuerpo** del pedido (`data=`), no en la URL — se usa típicamente para crear o enviar algo (un formulario, un registro nuevo).

Usamos `httpbin.org`, un servicio pensado justamente para probar requests: te devuelve, en JSON, exactamente lo que le mandaste.

In [ ]:
url_get = "http://httpbin.org/get"
url_post = "http://httpbin.org/post"
payload = {"name": "Ada", "curso": "Python"}

try:
    r_get = requests.get(url_get, params=payload, timeout=10)
    r_post = requests.post(url_post, data=payload, timeout=10)

    print("URL del GET (parámetros van en la URL):")
    print(r_get.url)
    print()
    print("Body del GET (no debería haber body):", r_get.request.body)
    print()
    print("Body del POST (los datos van en el cuerpo):", r_post.request.body)
    print()
    print("Lo que el servidor recibió como 'args' del GET:", r_get.json()["args"])
    print("Lo que el servidor recibió como 'form' del POST:", r_post.json()["form"])
except requests.exceptions.RequestException as e:
    print("No se pudo conectar a httpbin.org en este momento:", e)
    print("(La idea a llevarte: GET manda datos en la URL vía 'params', POST los manda en el cuerpo vía 'data'.)")

## Resumen: lo que te llevás de esta clase

- Toda API sigue el ciclo HTTP de **request** (pedido) y **response** (respuesta): método + URL de un lado, código de estado + headers + body del otro.
- **GET** para leer datos (con `params` opcionales); **POST** para enviar datos que el servidor va a procesar (con `data`).
- Revisá siempre `r.status_code` antes de confiar en `r.json()` — 2xx es éxito, 4xx es error tuyo, 5xx es error del servidor.
- `.json()` convierte el body en estructuras de Python. `pd.DataFrame()` alcanza si el JSON es plano; `pd.json_normalize()` lo aplana cuando viene anidado.
- Envolver los pedidos en `try/except requests.exceptions.RequestException` evita que un problema de red tire abajo todo el programa.

### Para seguir explorando

Buscá "free public APIs no auth needed" y probá con otra API que no requiera clave — el patrón (`requests.get()` → revisar `status_code` → `.json()` → `pd.DataFrame()` o `pd.json_normalize()`) se repite siempre.